# 06 — Risk-aversion sweep

**Sweep 3.** Fixed `n=8, K=2, p=3`. Vary `lambda` across two orders of
magnitude. At low `lambda` the problem is near-linear and greedy wins;
at high `lambda` it is covariance-dominated and frustrated. This sweep
tells us *when* QAOA's mechanism could matter.

In [ ]:
# === Bootstrap (Colab + local) ===
import sys, os, json
try:
    import google.colab  # noqa: F401
    get_ipython().system('test -d /content/fys5419 || git clone -q https://github.com/egil10/fys5419.git /content/fys5419')
    get_ipython().run_line_magic('cd', '/content/fys5419/project2/code/notebooks')
except ImportError:
    pass
sys.path.append('..')
from scripts.colab import setup; setup()

# === Project imports ===
from pathlib import Path
import numpy as np
import pandas as pd

from scripts.data      import load_returns
from scripts.portfolio import PortfolioProblem
from scripts.classical import brute_force, greedy_top_k, markowitz_round, simulated_annealing
from scripts.qaoa      import solve
from scripts.metrics   import prob_optimal

RESULTS = Path.cwd().parent / 'results'
RESULTS.mkdir(exist_ok=True)

In [ ]:
TICKERS = ['AAPL', 'MSFT', 'GOOGL', 'AMZN', 'NVDA', 'IBM', 'HON', 'ACN']
K, AP   = 2, 0.5
P       = 3
N_SEEDS = 10
START, END = '2023-01-01', '2025-12-31'

LAMBDAS = np.logspace(-1, 2, 7)  # 0.1, ..., 100

In [ ]:
r = load_returns(TICKERS, START, END, cache_name='n8_universe')
cache = RESULTS / 'risk_sweep.json'

if cache.exists():
    rows = json.loads(cache.read_text())
    print(f'loaded {cache.name}')
else:
    rows = []
    for lam in LAMBDAS:
        pf = PortfolioProblem(r.mu, r.Sigma, lam=float(lam), A=AP, K=K,
                              tickers=tuple(r.tickers))
        bf = brute_force(pf)

        sa_runs = [simulated_annealing(pf, n_sweeps=1000, seed=s) for s in range(N_SEEDS)]
        sa_med  = float(np.median([sr.cost for sr in sa_runs]))

        qres = solve(pf, p=P, n_restarts=N_SEEDS, seed=42)

        rows.extend([
            {'lambda': float(lam), 'solver': 'brute_force',     'cost': bf.cost,                  'ratio': 1.0},
            {'lambda': float(lam), 'solver': 'greedy_sharpe',   'cost': greedy_top_k(pf).cost,    'ratio': greedy_top_k(pf).cost / bf.cost},
            {'lambda': float(lam), 'solver': 'markowitz_round', 'cost': markowitz_round(pf).cost, 'ratio': markowitz_round(pf).cost / bf.cost},
            {'lambda': float(lam), 'solver': 'sa_median',       'cost': sa_med,                   'ratio': sa_med / bf.cost},
            {'lambda': float(lam), 'solver': f'qaoa_p{P}',      'cost': float(qres['energy']),    'ratio': float(qres['energy']) / bf.cost, 'p_optimal': prob_optimal(qres['probs'], bf.x)},
        ])
        print(f'  lambda={lam:6.2f}: brute={bf.cost:.4f}  qaoa={qres["energy"]:.4f}  p_opt={prob_optimal(qres["probs"], bf.x):.3f}')

    cache.write_text(json.dumps(rows, indent=2))
    print(f'saved -> {cache.name}')

pd.DataFrame(rows)